# Token Management Testing Notebook

This notebook provides functions to manage subscription plans and allocate tokens to users for testing purposes.

## Features:
- Create and manage subscription plans
- Create and manage clients
- Assign users to clients (which allocates tokens via subscription plans)
- Directly manage user token usage
- View current token allocations and usage

In [ ]:
# Setup: Import required modules
import sys
import os
from datetime import datetime, timedelta
from typing import Optional, List, Dict

# Add the backend directory to the path (adjust if running from different location)
backend_path = '/app/backend' if os.path.exists('/app/backend') else os.path.join(os.path.dirname(os.getcwd()), 'backend')
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

from open_webui.internal.db import get_db
from open_webui.models.subscriptions import (
    SubscriptionPlan, 
    Client, 
    UsagePerUser,
    SubscriptionPlanModel,
    ClientModel,
    UsagePerUserModel
)
from open_webui.models.users import User

print("✓ Imports successful!")

## User Management Functions

In [ ]:
def list_users() -> List[Dict]:
    """List all users in the system"""
    with get_db() as db:
        users = db.query(User).all()
        result = []
        for user in users:
            result.append({
                'id': user.id,
                'email': user.email,
                'name': user.name,
                'role': user.role
            })
        return result

def get_user_by_id(user_id: str) -> Optional[Dict]:
    """Get a specific user by ID"""
    with get_db() as db:
        user = db.query(User).filter(User.id == user_id).first()
        if user:
            return {
                'id': user.id,
                'email': user.email,
                'name': user.name,
                'role': user.role
            }
        return None

def get_user_by_email(email: str) -> Optional[Dict]:
    """Get a specific user by email"""
    with get_db() as db:
        user = db.query(User).filter(User.email == email).first()
        if user:
            return {
                'id': user.id,
                'email': user.email,
                'name': user.name,
                'role': user.role
            }
        return None

# Test: List all users
print("Available users:")
users = list_users()
for user in users:
    print(f"  - {user['name']} ({user['email']}) - ID: {user['id']}")

## Subscription Plan Management Functions

In [ ]:
def create_subscription_plan(
    plan_name: str,
    tokens_per_seat: int,
    description: Optional[str] = None,
    is_active: bool = True
) -> SubscriptionPlanModel:
    """
    Create a new subscription plan
    
    Args:
        plan_name: Unique name for the plan (e.g., "Basic", "Pro", "Enterprise")
        tokens_per_seat: Number of tokens allocated per seat
        description: Optional description of the plan
        is_active: Whether the plan is active
    
    Returns:
        SubscriptionPlanModel: The created plan
    """
    with get_db() as db:
        # Check if plan with same name exists
        existing = db.query(SubscriptionPlan).filter(
            SubscriptionPlan.plan_name == plan_name
        ).first()
        
        if existing:
            raise ValueError(f"Plan with name '{plan_name}' already exists (ID: {existing.id})")
        
        plan = SubscriptionPlan(
            plan_name=plan_name,
            tokens_per_seat=tokens_per_seat,
            description=description,
            is_active=is_active
        )
        db.add(plan)
        db.commit()
        db.refresh(plan)
        
        print(f"✓ Created subscription plan: {plan_name} (ID: {plan.id})")
        print(f"  Tokens per seat: {tokens_per_seat:,}")
        print(f"  Active: {is_active}")
        
        return SubscriptionPlanModel.model_validate(plan)

def list_subscription_plans() -> List[Dict]:
    """List all subscription plans"""
    with get_db() as db:
        plans = db.query(SubscriptionPlan).all()
        result = []
        for plan in plans:
            result.append({
                'id': plan.id,
                'plan_name': plan.plan_name,
                'tokens_per_seat': plan.tokens_per_seat,
                'description': plan.description,
                'is_active': plan.is_active,
                'created_at': plan.created_at
            })
        return result

def get_subscription_plan(plan_id: int) -> Optional[Dict]:
    """Get a subscription plan by ID"""
    with get_db() as db:
        plan = db.query(SubscriptionPlan).filter(SubscriptionPlan.id == plan_id).first()
        if plan:
            return {
                'id': plan.id,
                'plan_name': plan.plan_name,
                'tokens_per_seat': plan.tokens_per_seat,
                'description': plan.description,
                'is_active': plan.is_active
            }
        return None

def update_subscription_plan(
    plan_id: int,
    tokens_per_seat: Optional[int] = None,
    description: Optional[str] = None,
    is_active: Optional[bool] = None
) -> SubscriptionPlanModel:
    """Update an existing subscription plan"""
    with get_db() as db:
        plan = db.query(SubscriptionPlan).filter(SubscriptionPlan.id == plan_id).first()
        if not plan:
            raise ValueError(f"Plan with ID {plan_id} not found")
        
        if tokens_per_seat is not None:
            plan.tokens_per_seat = tokens_per_seat
        if description is not None:
            plan.description = description
        if is_active is not None:
            plan.is_active = is_active
        
        db.commit()
        db.refresh(plan)
        
        print(f"✓ Updated subscription plan: {plan.plan_name} (ID: {plan.id})")
        return SubscriptionPlanModel.model_validate(plan)

# Test: List existing plans
print("Existing subscription plans:")
plans = list_subscription_plans()
if plans:
    for plan in plans:
        print(f"  - {plan['plan_name']} (ID: {plan['id']}) - {plan['tokens_per_seat']:,} tokens/seat - Active: {plan['is_active']}")
else:
    print("  No plans found")

## Client Management Functions

In [ ]:
def create_client(
    name: str,
    subscription_plan_id: Optional[int] = None,
    seats_purchased: int = 1,
    next_reset_date: Optional[datetime] = None
) -> ClientModel:
    """
    Create a new client
    
    Args:
        name: Unique name for the client
        subscription_plan_id: ID of the subscription plan to assign (optional)
        seats_purchased: Number of seats purchased (default: 1)
        next_reset_date: Date when tokens will reset (optional, defaults to 30 days from now)
    
    Returns:
        ClientModel: The created client
    """
    with get_db() as db:
        # Check if client with same name exists
        existing = db.query(Client).filter(Client.name == name).first()
        if existing:
            raise ValueError(f"Client with name '{name}' already exists (ID: {existing.id})")
        
        # Validate subscription plan if provided
        if subscription_plan_id:
            plan = db.query(SubscriptionPlan).filter(
                SubscriptionPlan.id == subscription_plan_id
            ).first()
            if not plan:
                raise ValueError(f"Subscription plan with ID {subscription_plan_id} not found")
        
        # Set default reset date if not provided
        if next_reset_date is None:
            next_reset_date = datetime.utcnow() + timedelta(days=30)
        
        client = Client(
            name=name,
            subscription_plan_id=subscription_plan_id,
            seats_purchased=seats_purchased,
            next_reset_date=next_reset_date
        )
        db.add(client)
        db.commit()
        db.refresh(client)
        
        # Calculate total tokens if plan is assigned
        total_tokens = 0
        if subscription_plan_id and plan:
            total_tokens = plan.tokens_per_seat * seats_purchased
        
        print(f"✓ Created client: {name} (ID: {client.id})")
        print(f"  Seats: {seats_purchased}")
        if subscription_plan_id:
            print(f"  Plan: {plan.plan_name} (ID: {subscription_plan_id})")
            print(f"  Total tokens: {total_tokens:,} ({plan.tokens_per_seat:,} per seat × {seats_purchased} seats)")
        else:
            print(f"  No subscription plan assigned")
        
        return ClientModel.model_validate(client)

def list_clients() -> List[Dict]:
    """List all clients"""
    with get_db() as db:
        clients = db.query(Client).all()
        result = []
        for client in clients:
            plan_name = None
            tokens_per_seat = 0
            if client.subscription_plan:
                plan_name = client.subscription_plan.plan_name
                tokens_per_seat = client.subscription_plan.tokens_per_seat
            
            result.append({
                'id': client.id,
                'name': client.name,
                'subscription_plan_id': client.subscription_plan_id,
                'plan_name': plan_name,
                'tokens_per_seat': tokens_per_seat,
                'seats_purchased': client.seats_purchased,
                'total_tokens': tokens_per_seat * client.seats_purchased if plan_name else 0,
                'next_reset_date': client.next_reset_date
            })
        return result

def get_client(client_id: int) -> Optional[Dict]:
    """Get a client by ID"""
    with get_db() as db:
        client = db.query(Client).filter(Client.id == client_id).first()
        if client:
            plan_name = None
            tokens_per_seat = 0
            if client.subscription_plan:
                plan_name = client.subscription_plan.plan_name
                tokens_per_seat = client.subscription_plan.tokens_per_seat
            
            return {
                'id': client.id,
                'name': client.name,
                'subscription_plan_id': client.subscription_plan_id,
                'plan_name': plan_name,
                'tokens_per_seat': tokens_per_seat,
                'seats_purchased': client.seats_purchased,
                'total_tokens': tokens_per_seat * client.seats_purchased if plan_name else 0
            }
        return None

# Test: List existing clients
print("Existing clients:")
clients = list_clients()
if clients:
    for client in clients:
        print(f"  - {client['name']} (ID: {client['id']})")
        if client['plan_name']:
            print(f"    Plan: {client['plan_name']}, Seats: {client['seats_purchased']}, Total tokens: {client['total_tokens']:,}")
        else:
            print(f"    No plan assigned")
else:
    print("  No clients found")

## Token Allocation Functions

In [ ]:
def assign_user_to_client(user_id: str, client_id: int) -> UsagePerUserModel:
    """
    Assign a user to a client. This allocates tokens based on the client's subscription plan.
    
    Args:
        user_id: ID of the user to assign
        client_id: ID of the client to assign the user to
    
    Returns:
        UsagePerUserModel: The usage record for the user
    """
    with get_db() as db:
        # Verify user exists
        user = db.query(User).filter(User.id == user_id).first()
        if not user:
            raise ValueError(f"User with ID {user_id} not found")
        
        # Verify client exists
        client = db.query(Client).filter(Client.id == client_id).first()
        if not client:
            raise ValueError(f"Client with ID {client_id} not found")
        
        # Get or create usage record
        usage = db.query(UsagePerUser).filter(UsagePerUser.user_id == user_id).first()
        if usage:
            usage.client_id = client_id
            usage.updated_at = datetime.utcnow()
        else:
            usage = UsagePerUser(
                user_id=user_id,
                client_id=client_id,
                used_tokens=0
            )
            db.add(usage)
        
        db.commit()
        db.refresh(usage)
        
        # Calculate allocated tokens
        allocated_tokens = 0
        if client.subscription_plan:
            allocated_tokens = client.subscription_plan.tokens_per_seat * client.seats_purchased
        
        print(f"✓ Assigned user {user.name} ({user.email}) to client {client.name}")
        print(f"  Client ID: {client_id}")
        if client.subscription_plan:
            print(f"  Plan: {client.subscription_plan.plan_name}")
            print(f"  Allocated tokens: {allocated_tokens:,}")
            print(f"  Used tokens: {usage.used_tokens:,}")
            print(f"  Remaining tokens: {allocated_tokens - usage.used_tokens:,}")
        else:
            print(f"  Warning: Client has no subscription plan assigned")
        
        return UsagePerUserModel.model_validate(usage)

def set_user_tokens(user_id: str, used_tokens: int, client_id: Optional[int] = None) -> UsagePerUserModel:
    """
    Directly set the used_tokens for a user.
    Note: Setting used_tokens to 0 gives the user full quota.
    Setting to negative values gives extra tokens.
    
    Args:
        user_id: ID of the user
        used_tokens: Number of tokens used (set to 0 to reset, negative for extra tokens)
        client_id: Optional client ID to assign user to
    
    Returns:
        UsagePerUserModel: The usage record for the user
    """
    with get_db() as db:
        # Verify user exists
        user = db.query(User).filter(User.id == user_id).first()
        if not user:
            raise ValueError(f"User with ID {user_id} not found")
        
        # Get or create usage record
        usage = db.query(UsagePerUser).filter(UsagePerUser.user_id == user_id).first()
        if usage:
            usage.used_tokens = used_tokens
            if client_id is not None:
                usage.client_id = client_id
            usage.updated_at = datetime.utcnow()
        else:
            usage = UsagePerUser(
                user_id=user_id,
                client_id=client_id,
                used_tokens=used_tokens
            )
            db.add(usage)
        
        db.commit()
        db.refresh(usage)
        
        print(f"✓ Updated tokens for user {user.name} ({user.email})")
        print(f"  Used tokens: {used_tokens:,}")
        if client_id:
            print(f"  Client ID: {client_id}")
        
        return UsagePerUserModel.model_validate(usage)

def reset_user_tokens(user_id: str) -> UsagePerUserModel:
    """Reset a user's used_tokens to 0"""
    return set_user_tokens(user_id, 0)

def get_user_token_status(user_id: str, default_tokens: int = 100000) -> Dict:
    """
    Get comprehensive token status for a user
    
    Args:
        user_id: ID of the user
        default_tokens: Default tokens per user (from TOKEN_GATEWAY_DEFAULT_TOKENS)
    
    Returns:
        Dict with token information
    """
    with get_db() as db:
        user = db.query(User).filter(User.id == user_id).first()
        if not user:
            raise ValueError(f"User with ID {user_id} not found")
        
        usage = db.query(UsagePerUser).filter(UsagePerUser.user_id == user_id).first()
        
        allocated_tokens = default_tokens
        client_name = None
        plan_name = None
        
        if usage and usage.client_id:
            client = db.query(Client).filter(Client.id == usage.client_id).first()
            if client:
                client_name = client.name
                if client.subscription_plan:
                    plan_name = client.subscription_plan.plan_name
                    allocated_tokens = client.subscription_plan.tokens_per_seat * client.seats_purchased
        
        used_tokens = usage.used_tokens if usage else 0
        remaining_tokens = allocated_tokens - used_tokens
        
        return {
            'user_id': user_id,
            'user_name': user.name,
            'user_email': user.email,
            'used_tokens': used_tokens,
            'allocated_tokens': allocated_tokens,
            'remaining_tokens': remaining_tokens,
            'client_id': usage.client_id if usage else None,
            'client_name': client_name,
            'plan_name': plan_name,
            'percentage_used': (used_tokens / allocated_tokens * 100) if allocated_tokens > 0 else 0
        }

def list_all_user_tokens(default_tokens: int = 100000) -> List[Dict]:
    """List token status for all users"""
    users = list_users()
    result = []
    for user in users:
        try:
            status = get_user_token_status(user['id'], default_tokens)
            result.append(status)
        except Exception as e:
            result.append({
                'user_id': user['id'],
                'user_name': user['name'],
                'user_email': user['email'],
                'error': str(e)
            })
    return result

# Test: Show current token status for all users
print("Current token status for all users:")
token_statuses = list_all_user_tokens()
for status in token_statuses:
    if 'error' not in status:
        print(f"\n  {status['user_name']} ({status['user_email']})")
        print(f"    Used: {status['used_tokens']:,} / Allocated: {status['allocated_tokens']:,}")
        print(f"    Remaining: {status['remaining_tokens']:,} ({100 - status['percentage_used']:.1f}%)")
        if status['plan_name']:
            print(f"    Plan: {status['plan_name']} (Client: {status['client_name']})")
    else:
        print(f"\n  {status['user_name']} - Error: {status['error']}")

## Example Usage

### Example 1: Create a subscription plan and assign it to a user

In [ ]:
# Step 1: Create a subscription plan
plan = create_subscription_plan(
    plan_name="Basic Plan",
    tokens_per_seat=5000,  # 5k tokens per seat
    description="Professional plan with 500k tokens per seat",
    is_active=True
)

# Step 2: Create a client with this plan
# client = create_client(
#     name="Test Client",
#     subscription_plan_id=plan.id,
#     seats_purchased=1  # 1 seat = 500k tokens
# )

# Step 3: Assign a user to this client (replace with actual user ID)
# user_id = "your-user-id-here"
# assign_user_to_client(user_id, client.id)

### Example 2: Directly reset tokens for a user

In [ ]:
# Reset tokens for a user (sets used_tokens to 0)
# This gives them the full default quota (100k tokens by default)

# user_id = "your-user-id-here"
# reset_user_tokens(user_id)

# Or set to a specific value (negative values give extra tokens)
# set_user_tokens(user_id, -50000)  # Gives 50k extra tokens

### Example 3: Check token status for a specific user

In [ ]:
# Get detailed token status for a user
# user_id = "your-user-id-here"
# status = get_user_token_status(user_id)
# print(f"User: {status['user_name']}")
# print(f"Used: {status['used_tokens']:,} / Allocated: {status['allocated_tokens']:,}")
# print(f"Remaining: {status['remaining_tokens']:,}")

## Quick Reference

### Available Functions:

**User Management:**
- `list_users()` - List all users
- `get_user_by_id(user_id)` - Get user by ID
- `get_user_by_email(email)` - Get user by email

**Subscription Plans:**
- `create_subscription_plan(plan_name, tokens_per_seat, description=None, is_active=True)` - Create a plan
- `list_subscription_plans()` - List all plans
- `get_subscription_plan(plan_id)` - Get plan by ID
- `update_subscription_plan(plan_id, tokens_per_seat=None, description=None, is_active=None)` - Update plan

**Clients:**
- `create_client(name, subscription_plan_id=None, seats_purchased=1, next_reset_date=None)` - Create a client
- `list_clients()` - List all clients
- `get_client(client_id)` - Get client by ID

**Token Allocation:**
- `assign_user_to_client(user_id, client_id)` - Assign user to client (allocates tokens via plan)
- `set_user_tokens(user_id, used_tokens, client_id=None)` - Directly set used_tokens
- `reset_user_tokens(user_id)` - Reset user's tokens to 0
- `get_user_token_status(user_id, default_tokens=100000)` - Get detailed token status
- `list_all_user_tokens(default_tokens=100000)` - List token status for all users